# M1 lab — how DistilBERT tokenizes and encodes text

DistilBERT is a BERT-family **encoder** for understanding tasks. The tokenization steps illustrated here also explain the token-to-ID idea used by generative models, although their tokenizers and architectures may differ.

Run in Google Colab or in a local Python 3.11+ environment. Install `m1/requirements.txt` first. The first run downloads the public `distilbert-base-uncased` tokenizer from Hugging Face; no model weights, GPU, account or dataset are needed.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers>=4.45,<5"])


## 1. Load the tokenizer

Use the tokenizer associated with the exact checkpoint. BERT and DistilBERT use WordPiece subword tokenization.

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
print(type(tokenizer).__name__, "vocabulary size:", tokenizer.vocab_size)

## 2. From text to tokens and IDs

`input_ids` are vocabulary indices, not semantic values. `[CLS]` and `[SEP]` delimit the input, and `##` marks a continuation subword.

In [ ]:
text = "Tokenizing text is a core task of NLP."
encoded = tokenizer(text)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
for token, token_id in zip(tokens, encoded["input_ids"]):
    print(f"{token:>12}  {token_id}")
assert tokens[0] == "[CLS]" and tokens[-1] == "[SEP]"
assert len(tokens) == len(encoded["input_ids"])

## 3. Pad a batch and inspect the attention mask

The shorter sentence is padded to the length of the longer one. The mask has `1` for content and special tokens and `0` for padding.

In [ ]:
texts = ["A short sentence.", "Tokenizing unfamiliar words is useful in NLP."]
batch = tokenizer(texts, padding=True, truncation=True)
for sentence, ids, mask in zip(texts, batch["input_ids"], batch["attention_mask"]):
    print("text:", sentence)
    print("tokens:", tokenizer.convert_ids_to_tokens(ids))
    print("input_ids:", ids)
    print("attention_mask:", mask)
    assert len(ids) == len(mask)
    assert all((token_id == tokenizer.pad_token_id) == (keep == 0) for token_id, keep in zip(ids, mask))

## 4. Check the round trip

Decoding recovers readable text after removing the special tokens. Case and punctuation spacing may change because this checkpoint lowercases input.

In [ ]:
recovered = tokenizer.decode(encoded["input_ids"], skip_special_tokens=True)
print("original:", text)
print("decoded: ", recovered)
assert "tokenizing" in recovered.lower()

## What to observe

1. Which words were split into WordPiece subwords?
2. What changes if you replace the first sentence with a much longer one?
3. Why does the padded position have an attention-mask value of zero?
